# Generative AI and the Transformation of Music Information Ecosystems
## Main Analysis Notebook
### Case platforms: AOTY (Album of The Year) and RYM (RateYourMusic)

---

**Core research question**: When reviews themselves must be verified for authenticity, what is the value basis of an information service?

**Analytical approach**: institutional analysis with reproducible data methods

**Version**: v2.0 (international research edition)

### Research workflow
1. Web data collection (RYM + AOTY)
2. Time-series diagnostics (Welch + regression Chow + CUSUM + Bai-Perron-style segmentation)
3. Hybrid review-feature comparison (15 archived critic excerpts + 15 AI-style controls)
4. Uncalibrated trust-threshold scenario model
5. Analyst-coded platform scenario map
6. Figure generation with source notes (300 dpi)
7. Export results digest (statistics + figures)

## 1. Environment setup and dependencies

In [ ]:
# Install dependencies if needed (uncomment to run)
# !pip install -r ../requirements.txt

import sys
from pathlib import Path

# Make the src/ package importable
src_dir = Path.cwd().parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"[OK] src directory: {src_dir}")
print(f"[OK] Python version: {sys.version}")

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Academic serif font configuration (Times New Roman / SimSun)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'SimSun', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['axes.unicode_minus'] = False

print("[OK] core libraries imported")
print("[OK] fonts: serif (Times New Roman / SimSun)")

## 2. Run the full analysis pipeline

In [ ]:
# Option A: run the complete end-to-end pipeline (recommended)
from run_pipeline import ResearchPipeline

pipeline = ResearchPipeline()
pipeline.run()

# Analysis results are stored in pipeline.results
results = pipeline.results

## 3. Or run each analysis module individually

### 3.1 Data collection

In [ ]:
# RYM data collection
from data_collection.rym_scraper import RYMDataCollector

rym = RYMDataCollector(delay=2.0, use_cache=True)
rym_data = rym.generate_full_dataset()

for name, df in rym_data.items():
    print(f"[INFO] {name}: {len(df)} rows x {len(df.columns)} columns")

In [ ]:
# AOTY data collection
from data_collection.aoty_scraper import AOTYDataCollector

aoty = AOTYDataCollector(delay=2.0, use_cache=True)
aoty_data = aoty.generate_full_dataset()

for name, df in aoty_data.items():
    print(f"[INFO] {name}: {len(df)} rows x {len(df.columns)} columns")

### 3.2 Data preprocessing

In [ ]:
from preprocessing.data_preprocessing import DataPreprocessor

preprocessor = DataPreprocessor()
merged_data, quality_report = preprocessor.run_pipeline()

print(f"\n[INFO] merged dataset: {merged_data.shape}")
print(merged_data.head())

### 3.3 Time-series structural break analysis

In [ ]:
from analysis.structural_break_analysis import run_full_analysis

# Explicit synthetic benchmark: method demonstration, not platform evidence
rng = np.random.default_rng(42)
dates = pd.date_range('2020-01-01', '2026-07-01', freq='7D')
n = len(dates)

# Structural change at the ChatGPT release point
metric = np.where(
    dates < pd.Timestamp('2022-11-01'),
    3.5 + 0.3 * rng.standard_normal(n),
    3.2 + 0.5 * rng.standard_normal(n)  # mean drop, variance increase
)

test_data = pd.DataFrame({
    'date': dates,
    'avg_rating': np.clip(metric, 1, 5),
    'rating_count': rng.poisson(100, n).astype(float),
    'review_ratio': np.clip(0.3 + 0.1 * rng.standard_normal(n), 0.05, 0.6),
    'is_synthetic': True,
    'source_dataset': 'illustrative_structural_break_benchmark',
    'provenance_status': 'illustrative_simulation',
})

break_results = run_full_analysis(test_data, allow_non_empirical=True)

### 3.4 AI review detection and feature analysis

In [ ]:
from analysis.ai_review_analysis import AIReviewAnalyzer

analyzer = AIReviewAnalyzer()
ai_results = analyzer.run_full_analysis()

### 3.5 Trust threshold model

In [ ]:
from analysis.trust_threshold_analysis import TrustThresholdModel

model = TrustThresholdModel()
trust_results = model.full_analysis()
print('[INFO] Scenario model only; parameters are not calibrated and outputs are not forecasts.')

### 3.6 Platform competition analysis

In [ ]:
from analysis.platform_competition_analysis import CompetitiveAnalyzer

comp = CompetitiveAnalyzer()
comp_results = comp.run_full_analysis()

### 3.7 Visualization

In [ ]:
from visualization import generate_all_figures

generated = generate_all_figures(data=merged_data)
print(f"[OK] {len(generated)} figures generated")

### 3.8 Export results

The pipeline results are exported to a JSON file for downstream use. The
narrative research report is authored separately in `docs/Research_Report.md`.

In [ ]:
import json
from pathlib import Path

all_results = {
    "structural_break": break_results if 'break_results' in dir() else {},
    "ai_detection": ai_results if 'ai_results' in dir() else {},
    "trust_model": trust_results if 'trust_results' in dir() else {},
    "evidence_note": "Benchmark and scenario outputs are not platform findings",
    "competitive": comp_results if 'comp_results' in dir() else {},
}

export_path = Path("../data/processed/analysis_results.json")
export_path.parent.mkdir(parents=True, exist_ok=True)
with open(export_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)

print(f"[OK] Analysis results exported to {export_path}")